# 6. 원인 분석용 측정

2~5번 노트북은 "답이 얼마나 길어졌나"만 잰다. 이 노트북은 **왜 그런 결과가 나왔는지** 설명하는 데 쓸 수치를 잰다.
공격 사진(`adv/*.png`)은 읽기만 하고, 모델을 올리는 방법과 전처리는 2~5번과 같다.

재는 것은 여섯 가지다.

1. **가중치 변화**: 양자화가 그 부분의 숫자(가중치)를 얼마나 바꿨나. `변한 크기 ÷ 원래 크기`(%).
2. **그림 정보에 남은 공격**: 언어모델로 들어가는 그림 정보(연결부 출력)에서, 공격이 만든 변화가 양자화 뒤에도 같은 방향으로 남아 있나.
   - `공격 방향 보존`: 기준 모델에서 공격이 만든 변화와, 양자화 모델에서 공격이 만든 변화의 방향이 얼마나 같은가(1이면 완전히 같음, 0이면 무관).
   - `흔들림 비율`: 양자화가 그림 정보를 흔든 크기 ÷ 공격이 그림 정보를 바꾼 크기. 1보다 크면 양자화의 흔들림이 공격의 변화보다 크다.
3. **종료 점수 차**: 기준 모델이 공격 사진에 낸 답의 앞 {REF_LEN}토큰을 그대로 따라가게 하면서, 매 자리에서 `1등 토큰 점수 − 종료 토큰 점수`를 잰다.
   값이 클수록 종료가 멀고, 0이면 그 자리에서 종료 토큰이 1등이라 답이 끝난다.
4. **자기 답 기준 종료 점수 차**: 3번은 기준 모델의 답을 따라가게 한 값이다. 여기서는 각 모델이 공격 사진에 **스스로 낸 답**의 앞 16토큰에서 같은 점수 차를 잰다.
5. **원본 사진의 정상 종료 여유**: 기준 모델이 원본 사진에 낸 답을 그대로 따라가게 하고, 기준 모델이 답을 끝낸 바로 그 자리에서 `종료 토큰 점수 − 종료가 아닌 1등 점수`를 잰다.
   양수면 그 모델도 거기서 끝내고, 음수면 끝내지 않고 더 말한다.
6. **공격 사진에서 문장이 끝난 자리의 종료 여유**: 비전 인코더만 4비트로 양자화한 모델은 공격 사진에도 답을 끝낸다. 그 답을 다른 모델들에게 그대로 따라가게 하고,
   답이 끝난 바로 그 자리에서 5번과 같은 값을 잰다. 같은 문장을 다 말한 자리에서 다른 모델은 끝낼 수 있는지를 본다.

**실행 방법**
- 로컬(VSCode): 커널로 `.venv`를 고르고 전부 실행
- 터미널: `jupyter execute "6_원인분석_측정.ipynb" --inplace --timeout=-1`

**결과가 남는 곳**: 각 셀 아래의 표와 `results/6_원인분석_측정.csv`

In [ ]:
# 코랩에서만 패키지를 설치한다. 로컬은 README의 방법으로 만든 .venv를 쓴다.
import sys
if "google.colab" in sys.modules:
    !pip install -q -U transformers accelerate bitsandbytes
    !pip install -q --force-reinstall --no-cache-dir "pillow==11.2.1"

import torch, transformers, bitsandbytes, PIL
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| bitsandbytes", bitsandbytes.__version__, "| pillow", PIL.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

In [ ]:
MID = "HuggingFaceTB/SmolVLM-256M-Instruct"
Q = "Describe this image in one sentence."
MAX_NEW = 512

PARTS = {
    "V": ("비전 인코더", "model.vision_model"),
    "C": ("연결부",     "model.connector"),
    "L": ("언어모델",   "model.text_model"),
    "H": ("출력층",     "lm_head"),
}

CASES = [(None, None)] + [(p, b) for p in "VCLH" for b in (8, 4)]   # 기준 + 네 부분 × 8·4비트
REF_LEN = 64   # 종료 점수 차를 잴 때 따라가는 기준 답의 길이(토큰)
OWN_LEN = 16   # 자기 답 기준 종료 점수 차를 잴 때 보는 길이(토큰)

def case_name(part, bits):
    return "기준 bf16" if part is None else f"{PARTS[part][0]}만 {bits}비트"

In [ ]:
import os, gc, csv, logging
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

dev = "cuda"
proc = AutoProcessor.from_pretrained(MID); proc.image_processor.do_image_splitting = False
QUANT_TYPES = ("Linear4bit", "Linear8bitLt")

def load_case(part, bits):
    if part is None:
        return AutoModelForImageTextToText.from_pretrained(MID, dtype=torch.bfloat16).to(dev).eval()
    skip = [key for p, (_, key) in PARTS.items() if p != part]   # 대상이 아닌 부분은 전부 건너뛴다
    if bits == 8:
        cfg = BitsAndBytesConfig(load_in_8bit=True, llm_int8_skip_modules=skip)
    elif bits == 4:
        cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.bfloat16, llm_int8_skip_modules=skip)
    else:
        raise ValueError("bitsandbytes는 8비트와 4비트만 지원한다")
    return AutoModelForImageTextToText.from_pretrained(
        MID, quantization_config=cfg, dtype=torch.bfloat16, device_map=dev).eval()

def census(model):
    """부분별 [양자화된 Linear 수, 전체 Linear 수]"""
    c = {p: [0, 0] for p in PARTS}
    for name, m in model.named_modules():
        t = type(m).__name__
        if t != "Linear" and t not in QUANT_TYPES:
            continue
        owner = [p for p, (_, key) in PARTS.items() if name == key or name.startswith(key + ".")]
        assert len(owner) == 1, f"어느 부분인지 알 수 없는 Linear: {name}"
        c[owner[0]][1] += 1
        c[owner[0]][0] += t in QUANT_TYPES
    return c

def check_census(c, part):
    """의도한 부분만 전부 양자화됐는지 확인한다. 아니면 멈춘다."""
    for p, (q, total) in c.items():
        want = total if p == part else 0
        assert q == want, f"{PARTS[p][0]}: 양자화된 Linear {q}개 (기대 {want}개) → 건너뛸 이름을 확인할 것"

def load512(path):
    return Image.open(path).convert("RGB").resize((512, 512), Image.BICUBIC)

@torch.no_grad()
def gen_len(model, path):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": Q}]}]
    t = proc.apply_chat_template(msgs, add_generation_prompt=True)
    enc = proc(text=t, images=[load512(path)], return_tensors="pt").to(dev)
    out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False)
    return out.shape[1] - enc["input_ids"].shape[1]

In [ ]:
import bitsandbytes.functional as BF

EOS = proc.tokenizer.convert_tokens_to_ids("<end_of_utterance>")

def encode(path):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": Q}]}]
    t = proc.apply_chat_template(msgs, add_generation_prompt=True)
    enc = proc(text=t, images=[load512(path)], return_tensors="pt").to(dev)
    enc = {k: v for k, v in enc.items()}
    enc["pixel_values"] = enc["pixel_values"].to(torch.bfloat16)
    return enc

@torch.no_grad()
def run(model, enc):
    """한 번 계산하고 (연결부 출력 = 언어모델로 들어가는 그림 정보, 로짓)을 돌려준다."""
    box = {}
    hook = model.model.connector.register_forward_hook(
        lambda m, i, o: box.__setitem__("f", o[0] if isinstance(o, (tuple, list)) else o))
    out = model(**enc)
    hook.remove()
    return box["f"].float().flatten().cpu(), out.logits[0].float().cpu()

def with_answer(enc, ids):
    """질문 뒤에 정해진 답(ids)을 붙인 입력."""
    e = dict(enc)
    e["input_ids"] = torch.cat([enc["input_ids"], ids[None].to(dev)], dim=1)
    e["attention_mask"] = torch.ones_like(e["input_ids"])
    return e

def dequant(m):
    """양자화된 Linear의 가중치를 다시 실수로 풀어낸 값."""
    if type(m).__name__ == "Linear4bit":
        return BF.dequantize_4bit(m.weight.data, m.weight.quant_state).float().cpu()
    scb = m.weight.SCB if getattr(m.weight, "SCB", None) is not None else m.state.SCB
    return (m.weight.data.float() * scb.float().unsqueeze(1) / 127).cpu()

def weight_change(model, ref_w):
    """양자화된 층 전체에서 (변한 크기 ÷ 원래 크기)."""
    num = den = 0.0
    for name, m in model.named_modules():
        if type(m).__name__ in QUANT_TYPES:
            w0 = ref_w[name + ".weight"]
            num += (dequant(m) - w0).pow(2).sum().item()
            den += w0.pow(2).sum().item()
    return (num / den) ** 0.5 if den else 0.0

def cos(a, b):
    return torch.nn.functional.cosine_similarity(a, b, dim=0).item()

## 측정

먼저 기준 모델(bf16)에서 기준값을 만들고, 그다음 경우마다 같은 입력을 넣어 비교한다.

In [ ]:
files = sorted(f for f in os.listdir("imgs") if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")))
stems = [os.path.splitext(f)[0] for f in files]

# ── 기준 모델: 가중치, 그림 정보, 공격 사진에 대한 답의 앞부분 ──
model = load_case(None, None)
ref_w = {k: v.float().cpu() for k, v in model.state_dict().items() if k.endswith(".weight")}
ref = {}
for f, s in zip(files, stems):
    enc_c, enc_a = encode(f"imgs/{f}"), encode(f"adv/{s}.png")
    P = enc_a["input_ids"].shape[1]
    with torch.no_grad():
        ans = model.generate(**enc_a, max_new_tokens=REF_LEN, do_sample=False)[0, P:]
    ans = ans[ans != EOS]
    with torch.no_grad():
        clean_ans = model.generate(**enc_c, max_new_tokens=MAX_NEW, do_sample=False)[0, P:]
    assert clean_ans[-1] == EOS, f"{s}: 기준 모델이 원본 사진에서 답을 끝내지 않았다"
    f_c, _ = run(model, enc_c)
    f_a, _ = run(model, enc_a)
    ref[s] = dict(P=P, ans=ans.cpu(), clean_body=clean_ans[:-1].cpu(), f_c=f_c, f_a=f_a, shift=f_a - f_c,
                  shift_pct=(100 * (f_a - f_c).norm() / f_c.norm()).item())
    print(f"[기준] {s}: 공격이 그림 정보를 바꾼 크기 = 원래 크기의 {100 * (f_a - f_c).norm() / f_c.norm():.1f}%")
del model; gc.collect(); torch.cuda.empty_cache()

# ── 비전 인코더만 4비트 모델이 공격 사진에 낸 답(끝난 답만) ──
model = load_case("V", 4)
check_census(census(model), "V")
for f, s in zip(files, stems):
    enc_a = encode(f"adv/{s}.png")
    with torch.no_grad():
        full = model.generate(**enc_a, max_new_tokens=MAX_NEW, do_sample=False)[0, ref[s]["P"]:]
    ref[s]["v4_body"] = full[:-1].cpu() if full[-1] == EOS else None
    print(f"[비전 인코더만 4비트] {s}: 공격 사진의 답 {len(full)}토큰, 스스로 끝냄 = {full[-1] == EOS}")
del model; gc.collect(); torch.cuda.empty_cache()

# ── 경우마다 측정 ──
rows = []
for part, bits in CASES:
    name = case_name(part, bits)
    model = load_case(part, bits)
    check_census(census(model), part)
    wchg = weight_change(model, ref_w)
    for f, s in zip(files, stems):
        r = ref[s]
        enc_c, enc_a = encode(f"imgs/{f}"), encode(f"adv/{s}.png")
        f_c, _ = run(model, enc_c)
        f_a, logits = run(model, with_answer(enc_a, r["ans"]))
        lg = logits[r["P"] - 1:]                       # 답의 각 자리(다음 토큰을 고르는 자리)
        gap = lg.max(-1).values - lg[:, EOS]           # 1등 점수 − 종료 토큰 점수
        # 자기 답 기준: 이 모델이 공격 사진에 스스로 내는 답의 앞 OWN_LEN 토큰
        with torch.no_grad():
            own = model.generate(**enc_a, max_new_tokens=OWN_LEN, do_sample=False,
                                 output_logits=True, return_dict_in_generate=True)
        own_lg = torch.stack([x[0].float().cpu() for x in own.logits])
        own_gap = own_lg.max(-1).values - own_lg[:, EOS]
        # 원본 사진: 기준 모델이 답을 끝낸 자리에서 종료 토큰이 얼마나 앞서는가
        _, lg_c = run(model, with_answer(enc_c, r["clean_body"]))
        end = lg_c[-1].clone(); eos_score = end[EOS].item(); end[EOS] = float("-inf")
        # 공격 사진: 비전 인코더 4비트 모델이 답을 끝낸 자리에서 종료 토큰이 얼마나 앞서는가
        v4_margin = float("nan")
        if r["v4_body"] is not None:
            _, lg_v = run(model, with_answer(enc_a, r["v4_body"]))
            e2 = lg_v[-1].clone(); eos2 = e2[EOS].item(); e2[EOS] = float("-inf")
            v4_margin = eos2 - e2.max().item()
        rows.append(dict(case=name, part=part or "none", bits=bits or "bf16", image=s,
                         weight_change_pct=100 * wchg, attack_shift_pct=r["shift_pct"],
                         attack_dir_kept=cos(f_a - f_c, r["shift"]),
                         shake_ratio=((f_a - r["f_a"]).norm() / r["shift"].norm()).item(),
                         eos_gap_mean=gap.mean().item(), eos_gap_min=gap.min().item(),
                         own_gap_mean=own_gap.mean().item(), own_gap_min=own_gap.min().item(),
                         clean_end_margin=eos_score - end.max().item(),
                         adv_end_margin=v4_margin))
    print("측정 완료:", name)
    del model; gc.collect(); torch.cuda.empty_cache()

## 요약

경우별로 사진 3장의 값을 나란히 보여 준다(순서: 사진 이름순).

In [ ]:
def per_image(name, key, fmt):
    return " / ".join(fmt.format(r[key]) for r in rows if r["case"] == name)

cols = [("공격 방향 보존", "attack_dir_kept", "{:.2f}"), ("흔들림 비율", "shake_ratio", "{:.2f}"),
        ("종료 점수 차 평균", "eos_gap_mean", "{:.1f}"), ("종료 점수 차 최소", "eos_gap_min", "{:.1f}"),
        ("자기 답 종료 점수 차 평균", "own_gap_mean", "{:.1f}"), ("자기 답 종료 점수 차 최소", "own_gap_min", "{:.1f}"),
        ("원본 사진 정상 종료 여유", "clean_end_margin", "{:+.1f}"),
        ("공격 사진에서 문장이 끝난 자리의 종료 여유", "adv_end_margin", "{:+.1f}")]

print("사진 순서:", ", ".join(stems))
for part, bits in CASES:
    n = case_name(part, bits)
    w = next(r["weight_change_pct"] for r in rows if r["case"] == n)
    print(f"\n=== {n} | 가중치 변화 {w:.2f}%")
    for title, key, fmt in cols:
        print(f"    {title}: {per_image(n, key, fmt)}")

os.makedirs("results", exist_ok=True)
out_csv = "results/6_원인분석_측정.csv"
with open(out_csv, "w", newline="", encoding="utf-8-sig") as fp:
    w = csv.DictWriter(fp, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
print("\n저장:", out_csv)